# VateCon — Backend Tests
Tests complets pour l'API FastAPI : health, knowledge base, admin, agent IA et WebSocket.

> **Prérequis** : `docker compose up -d` doit être lancé avant d'exécuter ces tests.

In [ ]:
# Installer les dépendances de test
%pip install requests websocket-client httpx pytest colorama --quiet

In [ ]:
import requests
import json
import uuid
import time
import threading
import websocket
from colorama import Fore, Style, init
init(autoreset=True)

BASE_URL = "http://localhost:8000"
WS_URL  = "ws://localhost:8000"

results = []

def test(name, passed, detail=""):
    status = f"{Fore.GREEN}✓ PASS{Style.RESET_ALL}" if passed else f"{Fore.RED}✗ FAIL{Style.RESET_ALL}"
    results.append({"name": name, "passed": passed, "detail": detail})
    print(f"{status}  {name}" + (f"  →  {detail}" if detail else ""))

print("Base URL:", BASE_URL)
print("─" * 60)

## 1. Health Check

In [ ]:
r = requests.get(f"{BASE_URL}/health")
data = r.json()
test("GET /health — status 200", r.status_code == 200)
test("GET /health — status ok", data.get("status") == "ok", data.get("status"))
test("GET /health — service name", data.get("service") == "VateCon AI Support", data.get("service"))

## 2. API Docs

In [ ]:
r = requests.get(f"{BASE_URL}/docs")
test("GET /docs — OpenAPI UI accessible", r.status_code == 200)

r = requests.get(f"{BASE_URL}/openapi.json")
schema = r.json()
test("GET /openapi.json — schema valide", r.status_code == 200)
routes = list(schema.get("paths", {}).keys())
test("OpenAPI — routes présentes", len(routes) > 0, f"{len(routes)} routes")
print("  Routes:", routes)

## 3. Admin — Stats

In [ ]:
r = requests.get(f"{BASE_URL}/admin/stats")
test("GET /admin/stats — status 200", r.status_code == 200)
stats = r.json()
expected_keys = {"total_conversations", "escalated", "resolved", "today", "ai_resolution_rate"}
test("GET /admin/stats — clés présentes", expected_keys.issubset(stats.keys()), str(stats.keys()))
test("GET /admin/stats — ai_resolution_rate entre 0 et 100", 0 <= stats.get("ai_resolution_rate", -1) <= 100, str(stats.get("ai_resolution_rate")))
print("  Stats:", json.dumps(stats, indent=2))

## 4. Admin — Conversations

In [ ]:
r = requests.get(f"{BASE_URL}/admin/conversations")
test("GET /admin/conversations — status 200", r.status_code == 200)
convs = r.json()
test("GET /admin/conversations — retourne liste", isinstance(convs, list), f"{len(convs)} conversation(s)")

# Filtre par status
for status_filter in ["active", "escalated", "resolved"]:
    r = requests.get(f"{BASE_URL}/admin/conversations", params={"status": status_filter})
    test(f"GET /admin/conversations?status={status_filter}", r.status_code == 200, f"{len(r.json())} résultats")

## 5. Knowledge Base — Upload de documents

In [ ]:
# Test upload fichier TXT valide
txt_content = b"""VateCon Support FAQ

Q: What is VateCon?
A: VateCon is an AI-powered automated workforce marketplace that helps companies automate repetitive tasks.

Q: How do I reset my password?
A: Click on 'Forgot Password' on the login page and enter your email address.

Q: What are your pricing plans?
A: We offer Starter ($49/mo), Pro ($149/mo), and Enterprise (custom) plans.

Q: How do I contact support?
A: You can reach us at support@vatecon.com or via this chat interface.
"""

r = requests.post(
    f"{BASE_URL}/knowledge/upload",
    files={"file": ("faq.txt", txt_content, "text/plain")}
)
test("POST /knowledge/upload — TXT valide", r.status_code == 200, str(r.json()))
if r.status_code == 200:
    data = r.json()
    test("Upload — chunks indexés > 0", data.get("chunks_indexed", 0) > 0, f"{data.get('chunks_indexed')} chunks")

# Test doublon
r2 = requests.post(
    f"{BASE_URL}/knowledge/upload",
    files={"file": ("faq.txt", txt_content, "text/plain")}
)
test("Upload doublon — retourne 409", r2.status_code == 409, r2.json().get("detail", ""))

# Test type non supporté
r3 = requests.post(
    f"{BASE_URL}/knowledge/upload",
    files={"file": ("data.csv", b"col1,col2", "text/csv")}
)
test("Upload type invalide — retourne 400", r3.status_code == 400, r3.json().get("detail", ""))

## 6. Knowledge Base — FAQ

In [ ]:
faq_payload = {
    "entries": [
        {"question": "How do I cancel my subscription?", "answer": "Go to Settings > Billing > Cancel Subscription. No cancellation fees.", "category": "billing"},
        {"question": "Is my data secure?", "answer": "Yes, all data is encrypted at rest (AES-256) and in transit (TLS 1.3).", "category": "security"},
        {"question": "Do you offer a free trial?", "answer": "Yes, we offer a 14-day free trial with full access to all features.", "category": "sales"},
    ]
}

r = requests.post(
    f"{BASE_URL}/knowledge/faq",
    json=faq_payload,
    headers={"Content-Type": "application/json"}
)
test("POST /knowledge/faq — status 200", r.status_code == 200)
if r.status_code == 200:
    data = r.json()
    test("FAQ — entries_added == 3", data.get("entries_added") == 3, str(data))

# Liste documents
r = requests.get(f"{BASE_URL}/knowledge/documents")
test("GET /knowledge/documents — status 200", r.status_code == 200)
docs = r.json()
test("GET /knowledge/documents — retourne liste", isinstance(docs, list), f"{len(docs)} document(s)")
if docs:
    test("Document — champs corrects", all(k in docs[0] for k in ["id","filename","chunk_count","uploaded_at"]), str(docs[0].keys()))

## 7. WebSocket — Chat en temps réel

In [ ]:
session_id = str(uuid.uuid4())
received_messages = []
ws_connected = threading.Event()
ws_response = threading.Event()
conversation_id = [None]

def on_message(ws, message):
    data = json.loads(message)
    received_messages.append(data)
    print(f"  ← WS reçu: {data.get('type')} ", end="")
    if data.get("type") == "connected":
        conversation_id[0] = data.get("conversation_id")
        ws_connected.set()
        print(f"(conv_id: {conversation_id[0][:8]}...)")
    elif data.get("type") == "message":
        ws_response.set()
        print(f"(confidence: {data.get('confidence', 'N/A')})")
    else:
        print()

def on_error(ws, error):
    print(f"{Fore.RED}  WS error: {error}")
    ws_connected.set()
    ws_response.set()

ws = websocket.WebSocketApp(
    f"{WS_URL}/ws/chat/{session_id}",
    on_message=on_message,
    on_error=on_error,
)

ws_thread = threading.Thread(target=ws.run_forever, daemon=True)
ws_thread.start()

# Attendre la connexion
connected = ws_connected.wait(timeout=5)
test("WebSocket — connexion établie", connected)
test("WebSocket — message 'connected' reçu", any(m.get("type") == "connected" for m in received_messages))
test("WebSocket — conversation_id présent", conversation_id[0] is not None, conversation_id[0])

In [ ]:
# Envoyer un message au chat
ws_response.clear()
ws.send(json.dumps({"message": "What is VateCon and what are your pricing plans?"}))
print("  → Message envoyé au chat")

# Attendre la réponse (l'IA peut prendre quelques secondes)
got_response = ws_response.wait(timeout=30)
test("WebSocket — réponse IA reçue dans 30s", got_response)

ai_messages = [m for m in received_messages if m.get("type") == "message"]
if ai_messages:
    msg = ai_messages[-1]
    test("Réponse IA — contenu non vide", bool(msg.get("content")), f"{len(msg.get('content',''))} chars")
    test("Réponse IA — confidence présent", msg.get("confidence") is not None, str(msg.get("confidence")))
    test("Réponse IA — escalated présent", "escalated" in msg)
    print(f"  Réponse: {msg.get('content', '')[:200]}...")

In [ ]:
# Test escalade — mot-clé sensible
ws_response.clear()
ws.send(json.dumps({"message": "I want a refund immediately, this is fraud!"}))
got_escalation = ws_response.wait(timeout=30)
test("Escalade — réponse reçue", got_escalation)

escalation_msgs = [m for m in received_messages if m.get("type") == "message" and m.get("escalated")]
test("Escalade — conversation marquée escalated", len(escalation_msgs) > 0, f"{len(escalation_msgs)} message(s) escaladé(s)")
ws.close()

## 8. Admin — Vérifier la conversation créée

In [ ]:
if conversation_id[0]:
    r = requests.get(f"{BASE_URL}/admin/conversations/{conversation_id[0]}/messages")
    test("GET messages de la conversation — status 200", r.status_code == 200)
    msgs = r.json()
    test("Messages — liste non vide", len(msgs) > 0, f"{len(msgs)} message(s)")
    test("Messages — roles user/assistant présents",
         any(m["role"] == "user" for m in msgs) and any(m["role"] == "assistant" for m in msgs))
    
    # Résoudre la conversation
    r = requests.patch(f"{BASE_URL}/admin/conversations/{conversation_id[0]}/resolve")
    test("PATCH resolve conversation — status 200", r.status_code == 200)
    test("Resolve — status = resolved", r.json().get("status") == "resolved")
    
    # Conversation inexistante
    r = requests.get(f"{BASE_URL}/admin/conversations/fake-id/messages")
    test("Messages conv inexistante — retourne liste vide", r.status_code == 200 and r.json() == [])
else:
    print(f"{Fore.YELLOW}⚠ Skipped — pas de conversation_id (WebSocket non connecté)")

## 9. Résumé des tests

In [ ]:
passed = sum(1 for r in results if r["passed"])
failed = sum(1 for r in results if not r["passed"])
total  = len(results)

print("═" * 60)
print(f"RÉSULTATS : {Fore.GREEN}{passed} passés{Style.RESET_ALL} / {Fore.RED}{failed} échoués{Style.RESET_ALL} / {total} total")
print(f"Score     : {round(passed/total*100)}%" if total else "")
print("═" * 60)

if failed:
    print(f"\n{Fore.RED}Tests échoués :")
    for r in results:
        if not r["passed"]:
            print(f"  ✗ {r['name']}" + (f" → {r['detail']}" if r['detail'] else ""))